# Hyaline Kinase — annotate a structure in Colab

Give a **PDB ID** (e.g. `2hyy`, `1m17`) or a **kinase name** (e.g. `ABL1`, `EGFR`).
One run downloads the code, computes the interpretable geometric descriptors,
and returns the DFG state + inhibitor-class call with a rendered plot.

In [ ]:
# 1. Get the code + deps
!git clone --branch kinase-real-descriptors https://github.com/Varosync/Hyaline.git 2>/dev/null || echo 'already cloned'
%cd Hyaline
!pip -q install requests numpy scikit-learn pandas matplotlib pyarrow

In [ ]:
# 2. Your input: a PDB ID or a kinase name
INPUT = "2hyy"

In [ ]:
# 3. Analyze
import importlib.util, sys, requests, json
spec = importlib.util.spec_from_file_location('an', 'hyaline/kinase/analyze.py')
an = importlib.util.module_from_spec(spec); sys.modules['an'] = an; spec.loader.exec_module(an)

def resolve(x):
    # PDB IDs are 4 chars starting with a digit; kinase names start with a letter
    if len(x) == 4 and x[0].isdigit():
        return ('pdb', x)
    d = requests.get('https://klifs.net/api_v2/kinase_ID',
                     params={'kinase_name': x, 'species': 'Human'}, timeout=30).json()
    if not isinstance(d, list) or not d or not isinstance(d[0], dict):
        return ('pdb', x)  # fall back to treating it as a PDB code
    sl = requests.get('https://klifs.net/api_v2/structures_list',
                      params={'kinase_ID': [d[0]['kinase_ID']]}, timeout=30).json()
    best = sorted([s for s in sl if s.get('pocket')],
                  key=lambda s: s.get('quality_score', 0), reverse=True)[0]
    return ('klifs', best['structure_ID'])

kind, val = resolve(INPUT)
res = an.analyze(val) if kind == 'pdb' else an.analyze(int(val))
print(json.dumps(res.to_dict(), indent=2))

In [ ]:
# 4. Render: where this structure sits among real kinase conformations
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('research/kinase/data/kinase_descriptors.csv')
fig, ax = plt.subplots(figsize=(6.4, 4.6))
for lbl, c, y in [('DFG-in', '#0072B2', 0), ('DFG-out', '#E69F00', 1)]:
    m = df.y == y
    ax.scatter(df.distance[m], df.hinge_angle[m], s=30, c=c,
               edgecolors='white', linewidths=0.4, alpha=0.85, label=lbl)
ax.scatter([res.dfg_achelix_distance_A], [res.hinge_activation_angle_deg],
           s=340, marker='*', c='#111', edgecolors='white', zorder=5,
           label=f'{res.identifier} ({res.dfg_call})')
ax.set_xlabel('DFG-to-αC distance (Å)'); ax.set_ylabel('hinge angle (°)')
ax.set_title(f'{res.identifier}: {res.dfg_call} / {res.inhibitor_class}  (conf {res.dfg_confidence})')
ax.legend(); plt.show()